# Transformar Datos Departments
- Eliminar registros nulls del campo department_id
- Eliminar registros identicos
- Eliminar registros duplicados en base al campo created_timestamp
- Convertir columnas a asu tipo de dato correcto
- Escribir los datos transformados en el schema silver 

In [0]:
SELECT *FROM zenviro.bronze.v_departments 

# 1. Eliminar registros nulls del campo department_id

In [0]:
SELECT *FROM zenviro.bronze.v_departments WHERE department_id IS NOT NULL

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_departments 
AS
SELECT DISTINCT(*)
FROM zenviro.bronze.v_departments 
WHERE department_id IS NOT NULL
ORDER BY department_id


# 2. Eliminar registros identicos

In [0]:
SELECT DISTINCT(*) FROM zenviro.bronze.v_departments WHERE department_id IS NOT NULL ORDER BY department_id

# 3. Eliminar registros duplicados en base al campo created_timestamp

In [0]:
SELECT 
department_id,
MAX(created_timestamp)
FROM zenviro.bronze.v_departments 
GROUP BY department_id 

In [0]:
WITH cte
AS(
    SELECT 
    department_id,
    MAX(created_timestamp) AS max_timestamp
    FROM zenviro.bronze.v_departments 
    GROUP BY department_id
)
SELECT 
CAST(d.department_id AS BIGINT) AS department_id,
CAST(d.location_id AS BIGINT) AS location_id,
CAST(d.department_name AS STRING)AS department_name,
CAST(d.created_timestamp AS TIMESTAMP) AS created_timestamp

FROM tv_departments as d 
inner join cte
ON d.department_id = cte.department_id AND d.created_timestamp = cte.max_timestamp


# 4. Convertir columnas a asu tipo de dato correcto

In [0]:
WITH cte
AS(
    SELECT 
    department_id,
    MAX(created_timestamp) AS max_timestamp
    FROM zenviro.bronze.v_departments 
    GROUP BY department_id
)
SELECT 
CAST(d.department_id AS BIGINT) AS department_id,
CAST(d.location_id AS BIGINT) AS location_id,
CAST(d.department_name AS STRING)AS department_name,
CAST(d.created_timestamp AS TIMESTAMP) AS created_timestamp

FROM tv_departments as d 
inner join cte
ON d.department_id = cte.department_id AND d.created_timestamp = cte.max_timestamp

# 5. Escribir los datos transformados en el schema silver

In [0]:
CREATE OR REPLACE TABLE zenviro.silver.departments
AS
WITH cte
AS(
    SELECT 
    department_id,
    MAX(created_timestamp) AS max_timestamp
    FROM zenviro.bronze.v_departments 
    GROUP BY department_id
)
SELECT 
CAST(d.department_id AS BIGINT) AS department_id,
CAST(d.location_id AS BIGINT) AS location_id,
CAST(d.department_name AS STRING)AS department_name,
CAST(d.created_timestamp AS TIMESTAMP) AS created_timestamp

FROM tv_departments as d 
inner join cte
ON d.department_id = cte.department_id AND d.created_timestamp = cte.max_timestamp